In [1]:
import os
import random
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

###  Mushroom dataset
#### The mushroom_dataset file contain a class file which have 4 sub-files (classes):
- sub-file 1. conditionally_edible
- sub-file 2. deadly
- sub-file 3. edible
- sub-file 4. poisonous

#### Inside each sub-folder, it contains many mushroom files.

####  For the EDA, we create the following 3 classes to handle the data:
1. Class 1 - MushroomDataLoader is to create a data loader that will find and organize mushroom images.

2. Class 2 - MushroomAnalyzer is to perform statistical analysis and statistics.

3. Class 3 - MushroomVisualizer is to handle all the image visualization and display.

### Class 1

#### This class is to find, load, and organize the mushroom images from a dataset directory structure. 
#### It searches for image files using standard extensions and PIL - to classify them into the four mushroom safety categories,
#### i.e. edible, conditionally edible, poisonous, deadly based on folder names and filenames.
#### It displays statistics about the classified images.
#### It also displays folder structure if unclassified images are found to help with troubleshooting.

In [22]:
# Class 1

class MushroomDataLoader:
    
    # this function is to load and organize mushroom images from the dataset
    def __init__(self, dataset_path):
        self.dataset_path = Path(dataset_path)
        self.classes = ['edible', 'conditionally_edible', 'poisonous', 'deadly']
        
    def load_dataset(self):
        print("DATASET LOADING...")        
        return self._explore_dataset_structure()
    
    # this function is to check if the path exists
    def _explore_dataset_structure(self):
        if not self.dataset_path.exists():
            print(f"Error: Dataset path '{self.dataset_path}' does not exist!")
            print(f"Current working directory: {os.getcwd()}")
            print(f"Trying to list contents of current directory...")
            
            try:
                items = list(Path('.').iterdir())
                print(f"\nItems in current directory ({len(items)}):")
                for item in items[:20]:
                    print(f"  {item.name}{'/' if item.is_dir() else ''}")
                if len(items) > 20:
                    print(f"  ... and {len(items) - 20} more")
            except Exception as e:
                print(f"Error listing directory: {e}")
            return None
        
        # find all image files
        all_images = self._find_all_images()
        
        if len(all_images) == 0:
            print("\n*****   No images found with standard extensions.   *****")
            all_images = self._find_images_with_pil()
            print(f"\n *****   Found {len(all_images)} images using PIL detection.   *****")
        
        # classify images
        class_images = self._classify_images(all_images)
        
        # print statistics
        self._print_class_statistics(class_images, len(all_images))
        
        return class_images
    
    
    # this function is to find all the image files in the dataset
    def _find_all_images(self):

        image_extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG', 
                           '.bmp', '.BMP', '.tiff', '.TIFF']
        all_images = []
        
        for ext in image_extensions:
            all_images.extend(list(self.dataset_path.rglob(f"*{ext}")))
        
        print(f"\nTOTAL IMAGE FILES FOUND: {len(all_images)}")
        return all_images

    
    # this function is to find the images in the dataset using Python Imaging Library for non-standard extensions
    # PIL - gold-standard library to open, handle, and save many different image file formats in Python
    def _find_images_with_pil(self):
        all_images = []
        for item in self.dataset_path.rglob('*'):
            if item.is_file():
                try:
                    with Image.open(item) as img:
                        all_images.append(item)
                except:
                    pass
        return all_images
    
    # this function is to classify the images into different categories based on folder/filenames
    def _classify_images(self, all_images):
        class_images = {cls: [] for cls in self.classes}

        for img_path in all_images:
            img_str = str(img_path).lower()
            
            # classify based on folder names
            assigned = False
            for cls in self.classes:
                if cls in img_str:
                    # Check if it's not a partial match
                    if cls == 'edible' and 'conditionally_edible' in img_str:
                        continue
                    class_images[cls].append(img_path)
                    assigned = True
                    break
            
            # classify based on filename if not assigned
            if not assigned:
                filename = img_path.stem.lower()
                if any(word in filename for word in ['edible', 'safe', 'good', 'eatable']):
                    class_images['edible'].append(img_path)
                elif any(word in filename for word in ['condition', 'partial', 'maybe', 'cooked', 'boiled']):
                    class_images['conditionally_edible'].append(img_path)
                elif any(word in filename for word in ['poison', 'toxic', 'danger', 'toxic', 'inedible']):
                    class_images['poisonous'].append(img_path)
                elif any(word in filename for word in ['deadly', 'fatal', 'lethal', 'death', 'dead']):
                    class_images['deadly'].append(img_path)
                # Unclassified images are ignored
        
        return class_images
    

    # this function is to display statistics - classified images
    def _print_class_statistics(self, class_images, total_images):

        print("\nIMAGE COUNT BY CLASS (estimated):")
        total_classified = 0
        for cls in self.classes:
            count = len(class_images[cls])
            total_classified += count
            print(f"{cls:25s}: {count:4d} images")
        
        unclassified = total_images - total_classified
        if unclassified > 0:
            print(f"{'unclassified':25s}: {unclassified:4d} images")
            print(f"\nFound {unclassified} unclassified images.")
            self._show_folder_structure()
    
    # this function is to display folder structure - classification purpose
    def _show_folder_structure(self, max_depth=3):
        
        print(f"\n******  Folder structure (max depth {max_depth}):   *****")

        def print_tree(path, depth=0):
            if depth > max_depth:
                return
                
            indent = "  " * depth
            path_obj = Path(path)
            
            print(f"{indent}{path_obj.name}/")
            
            try:
                items = list(path_obj.iterdir())
                dirs = [d for d in items if d.is_dir()]
                files = [f for f in items if f.is_file()]
                
                for d in sorted(dirs):
                    print_tree(d, depth + 1)
                
                if files and depth < max_depth:
                    file_indent = "  " * (depth + 1)
                    for f in sorted(files)[:5]:
                        print(f"{file_indent}{f.name}")
                    if len(files) > 5:
                        print(f"{file_indent}... and {len(files)-5} more files")
                        
            except PermissionError:
                print(f"{indent}  [Permission denied]")
            except Exception as e:
                print(f"{indent}  [Error: {e}]")
        
        print_tree(self.dataset_path)


### Class 2 - MushroomAnalyzer

#### This class computes metrics on image dimensions, aspect ratios, class distributions, and format diversity. 
#### This helps us find the dataset issues like class imbalances and provides statistical visualizations for EDA.

In [48]:
# CLASS 2

class MushroomAnalyzer:    
    def __init__(self):
        self.class_colors = {
            'edible': 'green',
            'conditionally_edible': 'orange',
            'poisonous': 'red',
            'deadly': 'darkred'
        }
        self.classes = ['edible', 'conditionally_edible', 'poisonous', 'deadly']
    
    
    # this function is to analyze image statistics and create visualizations
    def analyze_statistics(self, class_images):
        print("\n **********   IMAGE STATISTICS ANALYSIS   **********")
        
        stats = self._calculate_statistics(class_images)
        
        if not stats['class_name']: 
            print("*****   No image data could be read for statistics.   *****")
            return None
        
        df_stats = pd.DataFrame(stats)
        self._print_statistics(df_stats)
        self._create_statistics_plots(df_stats, class_images)
        
        return df_stats
    
    
    # this private function is to calculate the image statistics for each class
    def _calculate_statistics(self, class_images):
        stats = {
            'class_name': [],
            'count': [],
            'avg_width': [],
            'avg_height': [],
            'min_width': [],
            'max_width': [],
            'min_height': [],
            'max_height': [],
            'aspect_ratios': []
        }
        
        for cls in self.classes:
            images = class_images.get(cls, [])
            if not images:
                continue
            
            widths, heights, aspect_ratios = self._process_images(images)
            
            if widths: 
                stats['class_name'].append(cls)
                stats['count'].append(len(images))
                stats['avg_width'].append(np.mean(widths))
                stats['avg_height'].append(np.mean(heights))
                stats['min_width'].append(np.min(widths))
                stats['max_width'].append(np.max(widths))
                stats['min_height'].append(np.min(heights))
                stats['max_height'].append(np.max(heights))
                stats['aspect_ratios'].append(aspect_ratios)
        
        return stats
    
    
    # this private function is to process a list of images and extract dimensions
    def _process_images(self, images):
        widths = []
        heights = []
        aspect_ratios = []
        
        for img_path in images:
            try:
                with Image.open(img_path) as img:
                    width, height = img.size
                    widths.append(width)
                    heights.append(height)
                    aspect_ratios.append(width / height)
            except Exception as e:
                print(f"Warning: Could not read {img_path.name}: {e}")
                continue
        
        return widths, heights, aspect_ratios
    
    
    # this private function is to print the statistics results of each class
    def _print_statistics(self, df_stats):
        print("\n CLASS STATISTICS:")
        for _, row in df_stats.iterrows():
            cls_name = row['class_name']
            print(f"\n{cls_name:25s}:")
            print(f"  Count: {row['count']:4d} images")
            print(f"  Average size: {row['avg_width']:.0f} x {row['avg_height']:.0f}")
            print(f"  Min size: {row['min_width']:.0f} x {row['min_height']:.0f}")
            print(f"  Max size: {row['max_width']:.0f} x {row['max_height']:.0f}")
    
    
    # this private function is to create graphs
    def _create_statistics_plots(self, df_stats, class_images):
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        # Bar chart of image counts
        self._plot_image_counts(axes[0, 0], df_stats)
        
        # Average image sizes
        self._plot_average_dimensions(axes[0, 1], df_stats)
        
        # Aspect ratio distribution
        self._plot_aspect_ratios(axes[1, 0], df_stats)
        
        # Resolution scatter plot
        self._plot_resolutions(axes[1, 1], class_images)
        
        plt.suptitle('Mushroom Dataset - Comprehensive Analysis', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig('mushroom_statistics.png', dpi=150, bbox_inches='tight')
        plt.show()
    

    # this private function is to display number of images
    def _plot_image_counts(self, ax, df_stats):
        ax.bar(df_stats['class_name'], df_stats['count'], 
               color=[self.class_colors.get(cls, 'gray') for cls in df_stats['class_name']])
        ax.set_title('Number of Images per Class', fontweight='bold')
        ax.set_ylabel('Count')
        ax.tick_params(axis='x', rotation=90)
        
        for i, v in enumerate(df_stats['count']):
            ax.text(i, v + max(df_stats['count']) * 0.01, str(v), 
                    ha='center', va='bottom', fontsize=9)
            
    
    # this private function is to display average dimensions
    def _plot_average_dimensions(self, ax, df_stats):
        bar_width = 0.35
        x = np.arange(len(df_stats))
        ax.bar(x - bar_width/2, df_stats['avg_width'], bar_width, 
               label='Width', color='skyblue')
        ax.bar(x + bar_width/2, df_stats['avg_height'], bar_width, 
               label='Height', color='lightcoral')
        ax.set_title('Average Image Dimensions', fontweight='bold')
        ax.set_ylabel('Pixels')
        ax.set_xticks(x)
        ax.set_xticklabels(df_stats['class_name'], rotation=90)
        ax.legend()
    
    
    # this private function display ratios aspect
    def _plot_aspect_ratios(self, ax, df_stats):
        aspect_ratio_data = []
        class_labels = []
        for i, cls in enumerate(df_stats['class_name']):
            aspect_ratio_data.extend(df_stats.loc[i, 'aspect_ratios'])
            class_labels.extend([cls] * len(df_stats.loc[i, 'aspect_ratios']))
        
        if aspect_ratio_data:
            aspect_df = pd.DataFrame({'class': class_labels, 'aspect_ratio': aspect_ratio_data})
            sns.boxplot(data=aspect_df, x='class', y='aspect_ratio', hue='class',
                       ax=ax, palette=self.class_colors, legend=False)
            ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Square (1:1)')
            ax.set_title('Aspect Ratio Distribution', fontweight='bold')
            ax.set_ylabel('Width / Height Ratio')
            ax.set_xlabel('')
            ax.tick_params(axis='x', rotation=45)
            ax.legend()
        else:
            ax.text(0.5, 0.5, 'No aspect ratio data', 
                   ha='center', va='center', fontsize=12)
            ax.axis('off')
    
    
    # this private function display resolutions
    def _plot_resolutions(self, ax, class_images):
        resolutions_by_class = {}
        for cls in self.classes:
            resolutions = []
            for img_path in class_images.get(cls, []):
                try:
                    with Image.open(img_path) as img:
                        resolutions.append(img.size)
                except:
                    continue
            if resolutions:
                resolutions_by_class[cls] = resolutions
        
        if resolutions_by_class:
            for cls, resolutions in resolutions_by_class.items():
                if resolutions:
                    widths, heights = zip(*resolutions)
                    ax.scatter(widths, heights, alpha=0.6, label=cls,
                              color=self.class_colors.get(cls, 'gray'), s=30)
            ax.set_title('Image Resolutions Scatter Plot', fontweight='bold')
            ax.set_xlabel('Width (pixels)')
            ax.set_ylabel('Height (pixels)')
            ax.legend()
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, 'No resolution data', 
                   ha='center', va='center', fontsize=12)
            ax.axis('off')
    
    
    # this private function is to create a summary report of the dataset
    def create_summary_report(self, class_images):
        print("\nDATASET SUMMARY REPORT")

        total_images = sum(len(images) for images in class_images.values())
        print(f"\nTOTAL IMAGES: {total_images}")
        
        if total_images == 0:
            print("\n ***** No images found in the dataset!   *****")
            return 0
        
        # Show class distribution
        print("\n**********   CLASS DISTRIBUTION:   **********")
        for cls in self.classes:
            count = len(class_images.get(cls, []))
            if total_images > 0:
                percentage = (count / total_images * 100)
            else:
                percentage = 0
            color_block = "█" * int(percentage / 5)
            print(f"{cls:25s}: {count:4d} images ({percentage:5.1f}%) {color_block}")
        
        # check for potential issues
        self._check_dataset_issues(class_images)
        
        # check image formats
        self._analyze_image_formats(class_images, total_images)
        
        return total_images
    
    
    # this private function is to check for the common dataset issues
    def _check_dataset_issues(self, class_images):
        print("\n **********   POTENTIAL ISSUES:   **********")
        issues_found = False
        
        # check for any empty classes
        for cls in self.classes:
            if len(class_images.get(cls, [])) == 0:
                print(f"\n - No images found for class: {cls}")
                issues_found = True
        
        # check for any class imbalances
        counts = [len(class_images.get(cls, [])) for cls in self.classes]
        non_zero_counts = [c for c in counts if c > 0]
        
        if non_zero_counts and len(non_zero_counts) > 1:
            imbalance_ratio = max(non_zero_counts) / min(non_zero_counts)
            if imbalance_ratio > 3:
                print(f"\n - Significant class imbalance detected (ratio: {imbalance_ratio:.1f})")
                issues_found = True
        
        if not issues_found:
            print("\n *****   No major issues detected in the dataset structure.   *****")

            
    # this private function is to analyze the distribution of image formats
    def _analyze_image_formats(self, class_images, total_images):
        print("\n **********   IMAGE FORMATS:   **********")
        formats = {}
        for cls, images in class_images.items():
            for img_path in images:
                ext = img_path.suffix.lower()
                formats[ext] = formats.get(ext, 0) + 1
        
        if formats:
            for fmt, count in sorted(formats.items()):
                percentage = (count / total_images * 100)
                print(f"{fmt:10s}: {count:4d} images ({percentage:5.1f}%)")
        else:
            print("\n *****   No image format information available   *****")


### Class 3 - MushroomVisualizer

#### The class creates and displays images of mushrooms from the dataset.
#### It generates three types of visualizations: 
1. organized sample images with class labels in a structured grid format, 
2. a compact overview grid showing multiple images per class, 
3. and detailed large-scale views of individual images with comprehensive metadata, 

#### All use color-coded borders to distinguish between the four mushroom safety categories (edible, conditionally edible, poisonous, deadly).

In [57]:
# CLASS 3

class MushroomVisualizer:

    
    def __init__(self):
        self.class_colors = {
            'edible': 'green',
            'conditionally_edible': 'orange',
            'poisonous': 'red',
            'deadly': 'darkred'
        }
        self.classes = ['edible', 'conditionally_edible', 'poisonous', 'deadly']
    

    # this function is to display sample images from each class
    def display_sample_images(self, class_images, num_samples=4):

        print("\n**********   SAMPLE IMAGES FROM EACH CLASSIFICATION   **********")
        
        non_empty_classes = [cls for cls in self.classes if class_images.get(cls)]
        
        if not non_empty_classes:
            print("\n*****   No classified images found!   *****")
            return
        
        fig, axes = plt.subplots(len(non_empty_classes), num_samples + 1, 
                                figsize=(4 * (num_samples + 1), 3 * len(non_empty_classes)))
        
        if len(non_empty_classes) == 1:
            axes = axes.reshape(1, -1)
        
        for row_idx, cls in enumerate(non_empty_classes):
            images = class_images.get(cls, [])
            print(f"\nClass: {cls} - Found {len(images)} images")
            
            sample_images = self._select_sample_images(images, num_samples)
            self._plot_class_row(axes, row_idx, cls, sample_images)
        
        plt.suptitle("\n\n\n Mushroom Dataset - Sample Images by Classification", 
                    fontsize=18, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig('mushroom_samples.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        return fig
    
    #this private function is to select sample images for display
    def _select_sample_images(self, images, num_samples):

        if len(images) >= num_samples:
            return random.sample(images, num_samples)
        else:
            if len(images) > 0:
                print(f"\n*****   (Only {len(images)} images available, showing all)   *****")
            return images
    
    # this private function is to plot a row of images for one particular class
    def _plot_class_row(self, axes, row_idx, cls, sample_images):
        # Show class name in first column
        axes[row_idx, 0].text(0.5, 0.5, cls.upper(), 
                             fontsize=16, fontweight='bold',
                             color=self.class_colors.get(cls, 'black'),
                             horizontalalignment='center',
                             verticalalignment='center',
                             transform=axes[row_idx, 0].transAxes)
        axes[row_idx, 0].axis('off')
        
        # using for-loop to display sample images
        for col_idx, img_path in enumerate(sample_images, 1):
            self._plot_single_image(axes[row_idx, col_idx], img_path, cls, col_idx)
        
        # using for-loop to hide empty subplots
        for col_idx in range(len(sample_images) + 1, axes.shape[1]):
            axes[row_idx, col_idx].axis('off')
    
    # this private function is to display a single image with metadata
    def _plot_single_image(self, ax, img_path, cls, img_idx):
        try:
            img = Image.open(img_path)
            
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            # here to resize if too large
            max_size = 300
            if max(img.size) > max_size:
                ratio = max_size / max(img.size)
                new_size = (int(img.size[0] * ratio), int(img.size[1] * ratio))
                img = img.resize(new_size, Image.Resampling.LANCZOS)
            
            ax.imshow(img)
            
            filename = img_path.name
            if len(filename) > 20:
                filename = filename[:17] + "..."
            
            ax.set_title(f"{filename}\n{img.size[0]}x{img.size[1]}", fontsize=9)
            
            # here to color-coded border
            border_color = self.class_colors.get(cls, 'gray')
            for spine in ax.spines.values():
                spine.set_edgecolor(border_color)
                spine.set_linewidth(3)
            
            ax.axis('off')
            
            print(f"   Image {img_idx}: {img_path.name} ({img.size[0]}x{img.size[1]}, {img.mode})")
            
        except Exception as e:
            print(f"\n *****   Error loading {img_path.name}: {str(e)}   *****")
            ax.text(0.5, 0.5, "Error\nloading\nimage",
                   horizontalalignment='center',
                   verticalalignment='center',
                   fontsize=10)
            ax.axis('off')
    
    # this function is to display a grid of images from all classes
    def display_image_grid(self, class_images, images_per_class=8, grid_cols=4):
        print("\n*****   IMAGE GRID - ALL CLASSES   *****")
        
        non_empty_classes = [cls for cls in self.classes if class_images.get(cls)]
        
        if not non_empty_classes:
            print("\n*****   No classified images found!   *****")
            return
        
        grid_rows = len(non_empty_classes)
        fig, axes = plt.subplots(grid_rows, grid_cols, 
                                figsize=(4 * grid_cols, 3 * grid_rows))
        
        if grid_rows == 1:
            axes = axes.reshape(1, -1)
        
        for row_idx, cls in enumerate(non_empty_classes):
            images = class_images.get(cls, [])
            selected_images = self._select_grid_images(images, images_per_class)
            self._plot_grid_row(axes, row_idx, cls, selected_images, grid_cols)
        
        plt.suptitle("\n\n\n Mushroom Classification Dataset Overview", 
                    fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig('mushroom_grid.png', dpi=150, bbox_inches='tight')
        plt.show()
    
    # this private function is to select images for grid display
    def _select_grid_images(self, images, images_per_class):
        if len(images) >= images_per_class:
            return random.sample(images, images_per_class)
        else:
            return images
    
    # this private function is to display a row in the image grid
    def _plot_grid_row(self, axes, row_idx, cls, selected_images, grid_cols):
        for col_idx in range(grid_cols):
            if col_idx < len(selected_images):
                img_path = selected_images[col_idx]
                self._plot_grid_image(axes[row_idx, col_idx], img_path, cls)
            else:

                # here to fill empty spots
                if col_idx == 0:
                    axes[row_idx, col_idx].text(0.5, 0.5, cls.upper(),
                                               fontsize=12, fontweight='bold',
                                               color=self.class_colors.get(cls, 'black'),
                                               horizontalalignment='center',
                                               verticalalignment='center')
            axes[row_idx, col_idx].axis('off')

    # this private function is to plot grid image
    def _plot_grid_image(self, ax, img_path, cls):
        try:
            img = Image.open(img_path)
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            max_size = 200
            if max(img.size) > max_size:
                ratio = max_size / max(img.size)
                new_size = (int(img.size[0] * ratio), int(img.size[1] * ratio))
                img = img.resize(new_size, Image.Resampling.LANCZOS)
            
            ax.imshow(img)
            
            filename = img_path.stem
            if len(filename) > 10:
                filename = filename[:8] + ".."
            
            ax.set_title(f"{cls[:10]}\n{filename}", 
                       fontsize=9, color=self.class_colors.get(cls, 'black'))
            
            border_color = self.class_colors.get(cls, 'gray')
            for spine in ax.spines.values():
                spine.set_edgecolor(border_color)
                spine.set_linewidth(2)
            
        except Exception as e:
            ax.text(0.5, 0.5, "Error", 
                   horizontalalignment='center',
                   verticalalignment='center',
                   fontsize=10)
    
    # this function is to display larger versions of sample images with more information
    def display_large_samples(self, class_images, num_samples=3):
        print("\n ***** DETAILED IMAGE SAMPLES *****")
        
        for cls in self.classes:
            images = class_images.get(cls, [])
            if not images:
                print(f"\n !!!!! No images found for class: {cls} !!!!!")
                continue
            
            samples = self._select_large_samples(images, num_samples)
            self._display_class_samples(cls, samples)

    # this private function is to select samples for large display
    def _select_large_samples(self, images, num_samples):
        if len(images) >= num_samples:
            return random.sample(images, num_samples)
        else:
            return images

    # this private function is to display large samples for a particular class
    def _display_class_samples(self, cls, samples):
        print(f"\nCLASS: {cls.upper()}")
        
        fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 5))
        if len(samples) == 1:
            axes = [axes]
        
        for idx, (img_path, ax) in enumerate(zip(samples, axes)):
            self._plot_large_image(ax, img_path, cls)
        
        plt.suptitle(f"Mushroom Class: {cls.upper()}", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print(f"Class Color: {self.class_colors.get(cls, 'N/A')}")
        print(f"Total images in class: {len(samples)}")
        print(f"Sample images shown: {len(samples)}")
    
    # this private function is to plot a large image with detailed metadata
    def _plot_large_image(self, ax, img_path, cls):
        try:
            img = Image.open(img_path)
            
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            ax.imshow(img)
            
            file_size = os.path.getsize(img_path) / 1024  # KB
            img_info = f"{img_path.name}\n"
            img_info += f"Size: {img.size[0]} × {img.size[1]}\n"
            img_info += f"Mode: {img.mode}\n"
            img_info += f"Filesize: {file_size:.1f} KB\n"
            img_info += f"Path: {os.path.relpath(img_path, img_path.parent.parent)}"
            
            ax.set_title(img_info, fontsize=9)
            
            border_color = self.class_colors.get(cls, 'gray')
            for spine in ax.spines.values():
                spine.set_edgecolor(border_color)
                spine.set_linewidth(4)
            
            ax.axis('off')
            
        except Exception as e:
            ax.text(0.5, 0.5, f"Error loading image:\n{str(e)}",
                   horizontalalignment='center', verticalalignment='center',
                   fontsize=10)
            ax.axis('off')


In [ ]:
# MAIN FUNCTION 

def main():
    print("MUSHROOM DATASET EDA")
    
    # Find dataset path
    dataset_path = "."
    possible_paths = [
        "mushroom-classification-dataset",
        "dataset",
        "data",
        "mushroom_data",
        "mushrooms",
        "."
    ]
    
    for path in possible_paths:
        if Path(path).exists():
            dataset_path = path
            print(f"\nLOADING... ")
            break
    
    # component initializations
    loader = MushroomDataLoader(dataset_path)
    analyzer = MushroomAnalyzer()
    visualizer = MushroomVisualizer()
    
    # load dataset
    class_images = loader.load_dataset()
    
    if class_images and any(class_images.values()):
        print("\n **********   DATASET ANALYSIS   **********")
        
        # Run analysis pipeline
        analyzer.create_summary_report(class_images)
        analyzer.analyze_statistics(class_images)
        visualizer.display_sample_images(class_images, num_samples=4)
        visualizer.display_image_grid(class_images, images_per_class=8, grid_cols=4)
        visualizer.display_large_samples(class_images, num_samples=3)
    
    return loader, analyzer, visualizer, class_images


if __name__ == "__main__":
    loader, analyzer, visualizer, class_images = main()

MUSHROOM DATASET EDA

LOADING... 
DATASET LOADING...

TOTAL IMAGE FILES FOUND: 102716

IMAGE COUNT BY CLASS (estimated):
edible                   : 18417 images
conditionally_edible     : 61968 images
poisonous                : 21138 images
deadly                   : 1190 images
unclassified             :    3 images

Found 3 unclassified images.

******  Folder structure (max depth 3):   *****
/
  .ipynb_checkpoints/
    Untitled-checkpoint.ipynb
    mushroomEDA - detailed-checkpoint.ipynb
    mushroomEDA v1-checkpoint.ipynb
    mushroomEDA-checkpoint.ipynb
  mushroom_dataset/
    Classes/
      .ipynb_checkpoints/
      conditionally_edible/
      deadly/
      edible/
      poisonous/
      .DS_Store
    .DS_Store
  .DS_Store
  Questions.pages
  Team 8 - Milestone 1_ Project Abstract.pdf
  Team Contract - Google Docs.pdf
  Untitled.ipynb
  ... and 7 more files

 **********   DATASET ANALYSIS   **********

DATASET SUMMARY REPORT

TOTAL IMAGES: 102713

**********   CLASS DISTRIBUTION: